# Python for MATLAB users · 7 · Errors and debugging

Catching errors, reading a traceback, and the debugger, on a function ported from MATLAB.

**Run this cell first** (Shift+Enter). It downloads the project and installs it
into the Python of this notebook. Every notebook runs on its own machine, so every
notebook starts with this cell, and needs it again when Colab gives it a new machine.

In [ ]:
!test -d /content/pdm-workshop || git clone -q --depth 1 https://github.com/alessandro-rubin-sate/pdm-workshop.git /content/pdm-workshop
%cd /content/pdm-workshop
%pip install -q .

Files you edit while this notebook runs must be read again before they are used.
autoreload does that, before every cell:

In [ ]:
%load_ext autoreload
%autoreload 2

## try and except

### Idiom 11 - try / except: handle the error you expect.

`idioms/11_try_except.py`

```
MATLAB:  try ... catch ME ... end, and error("...")
```

In [ ]:
from pdm_workshop.reports import load_feature_table, machine_report

table = load_feature_table()

In [ ]:
# Catch the error you expect, by its type, and carry on
for machine in ["M07", "M99", "M03"]:
    try:
        report = machine_report(table, machine, hours=40)
    except ValueError as error:
        print(f"skipped {machine}: {error}")
    else:
        print(f"{machine}: {report.height} rows")

In [ ]:
# finally runs in every case: the place for cleanup
def risky(value):
    try:
        return 10 / value
    except ZeroDivisionError:
        return float("inf")
    finally:
        print(f"  finished with {value}")


print(risky(4))
print(risky(0))

In [ ]:
# Raising an error yourself: say what was wrong and what is allowed
def check_width(width):
    if width <= 0:
        raise ValueError(f"width must be positive, got {width}")
    return width


try:
    check_width(-5)
except ValueError as error:
    print("ValueError:", error)

In [ ]:
# A bare `except:` would also hide this typo. Catching ValueError does not.
try:
    try:
        machine_reprot(table, "M07")
    except ValueError:
        print("never printed")
except NameError as error:
    print("NameError got through, as it should:", error)

## The exercise: the feature that changes the data

**Stage 1.** Run the script and read the traceback from the bottom: the last
line says what, the block above it says where.

In [ ]:
!python exercises/run_exercise.py

The same script, run inside the notebook with `%run`. After an error, `%debug`
opens the debugger at the line that failed: try `p i`, `p n`, `p out.shape`, then `q`.

In [ ]:
%run exercises/run_exercise.py

In [ ]:
%debug

| In MATLAB | In the debugger (ipdb) |
| --- | --- |
| `keyboard` | `breakpoint()` in the code |
| `dbstop if error` | `%pdb on`, or `%debug` after the error |
| Look at a variable | `p name`, or `pp name` for long values |
| F10, F11, F5 | `n` next line, `s` step into, `c` continue |
| `dbup`, `dbdown` | `u`, `d` |
| `dbquit` | `q` |

To fix the file: open the **Files** panel (the folder icon on the left), go to
`pdm-workshop/exercises`, double-click `legacy_features.py`. It opens in an editor
next to the notebook. Correct the line, save with Ctrl+S, and run `%run` again.

**Stage 2.** The script runs, and the numbers are wrong: the sensor bias is about
3 before the RMS is computed and 0 after. Put `breakpoint()` on the first line inside
the loop of `window_rms`, run the cell below, and watch `signal` while stepping
with `n`. Two traps seen earlier are combined here: name them before fixing anything.

In [ ]:
%run exercises/run_exercise.py

When both stages are fixed, the two tests pass:

In [ ]:
!python -m pytest -q exercises

The explanation is in `exercises/SOLUTION.md`: open it at the end.